In [0]:
# Load Bronze layer data
df_bronze = spark.table("workspace.redditrecon.posts_bronze")

print(f"📥 Loaded {df_bronze.count():,} records from Bronze layer")
print(f"   Table: workspace.redditrecon.posts_bronze\n")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType, LongType, TimestampType, BooleanType

print("🔍 VALIDATING BRONZE DATA TYPES & SCHEMA")
print("="*80)

# Verify schema
print("\n📋 Bronze Schema:")
df_bronze.printSchema()

# Validate expected data types
expected_types = {
    "id": StringType(),
    "author": StringType(),
    "subreddit": StringType(),
    "title": StringType(),
    "score": LongType(),
    "created_at": TimestampType(),
    "over_18": BooleanType()
}

print("\n✅ Validating critical field types...")
for field_name, expected_type in expected_types.items():
    actual_field = [f for f in df_bronze.schema.fields if f.name == field_name][0]
    if type(actual_field.dataType) == type(expected_type):
        print(f"   ✓ {field_name}: {actual_field.dataType}")
    else:
        print(f"   ✗ {field_name}: Expected {expected_type}, got {actual_field.dataType}")

print("\n✅ Schema validation complete")
print("="*80)

In [0]:
print("\n🧹 STANDARDIZATION - CLEAN & NORMALIZE TEXT")
print("="*80)

# Clean and normalize text fields
df_standardized = df_bronze \
    .withColumn("author", F.trim(F.col("author"))) \
    .withColumn("subreddit", F.trim(F.col("subreddit"))) \
    .withColumn("title", F.trim(F.col("title"))) \
    .withColumn("selftext", F.trim(F.col("selftext"))) \
    .withColumn("url", F.trim(F.col("url"))) \
    .withColumn("link_flair_text", F.trim(F.col("link_flair_text"))) \
    .withColumn("author_flair_text", F.trim(F.col("author_flair_text")))

print("✅ Standardization complete - text fields cleaned and trimmed")
print(f"   Records processed: {df_standardized.count():,}")
print("="*80)

In [0]:
print("\n⚠️ CHECKING FOR NULL VALUES")
print("="*80)

# Check critical fields for nulls
critical_fields = ["id", "author", "title", "created_at", "score"]

null_counts = df_standardized.select(
    [F.sum(F.when(F.col(field).isNull() | (F.trim(F.col(field)) == ""), 1).otherwise(0)).alias(field) 
     for field in critical_fields]
).collect()[0]

print("\n📊 Null/Empty Values in Critical Fields:")
total_nulls = 0
for field in critical_fields:
    null_count = null_counts[field]
    total_nulls += null_count
    status = "✓" if null_count == 0 else "✗"
    print(f"   {status} {field}: {null_count:,} nulls")

if total_nulls == 0:
    print("\n✅ No null values found in critical fields")
else:
    print(f"\n⚠️ Total null/empty values found: {total_nulls:,}")
    
print("="*80)

In [0]:
print("\n🤖 BOT DETECTION - FLAGGING BOT ACCOUNTS")
print("="*80)

# Only flag obvious moderator/auto bots - NOT general words like "robot"
# Conservative approach: automoderator, bot, [deleted], [removed], moderator, automod
df_with_bot_flag = df_standardized.withColumn(
    "is_bot",
    F.when(
        F.lower(F.col("author")).rlike(
            "(automoderator|bot|^\\[deleted\\]$|^\\[removed\\]$|moderator|automod)"
        ),
        True
    ).otherwise(False)
)

# Count bot records
bot_stats = df_with_bot_flag.groupBy("is_bot").count().collect()
bot_count = [row['count'] for row in bot_stats if row['is_bot'] == True]
bot_count = bot_count[0] if bot_count else 0
total_count = df_with_bot_flag.count()

print(f"\n📊 Bot Detection Results:")
print(f"   Bot accounts flagged:     {bot_count:,} ({bot_count/total_count*100:.2f}%)")
print(f"   Regular accounts:         {total_count - bot_count:,} ({(total_count-bot_count)/total_count*100:.2f}%)")
print("\n✅ Bot detection complete (conservative - moderator bots only)")
print("   Note: Words like 'robot' are NOT flagged as bots")
print("="*80)

In [0]:
print("\n🚨 SPAM DETECTION - BASIC CHECKS")
print("="*80)

# Basic spam indicators:
# 1. Excessive repetition in title (same word repeated many times)
# 2. Very short titles with suspicious patterns
# 3. URLs with excessive parameters (spam links)

# Check for spam patterns
spam_checks = df_with_bot_flag.select(
    F.col("id"),
    F.col("title"),
    F.col("author"),
    # Count repeated characters (e.g., "AAAAAA" or "!!!!!!")
    F.when(F.col("title").rlike("(.)\\1{10,}"), True).otherwise(False).alias("has_char_spam"),
    # Check for excessive special characters
    F.when(F.length(F.regexp_replace(F.col("title"), "[^!@#$%^&*()]", "")) > 10, True).otherwise(False).alias("excessive_special_chars"),
    # Very high score posts (potential vote manipulation)
    F.when(F.col("score") > 100000, True).otherwise(False).alias("suspiciously_high_score")
)

spam_counts = spam_checks.agg(
    F.sum(F.when(F.col("has_char_spam"), 1).otherwise(0)).alias("char_spam"),
    F.sum(F.when(F.col("excessive_special_chars"), 1).otherwise(0)).alias("special_char_spam"),
    F.sum(F.when(F.col("suspiciously_high_score"), 1).otherwise(0)).alias("high_score_spam")
).collect()[0]

print("\n📊 Spam Detection Results:")
print(f"   Character repetition spam:    {spam_counts['char_spam']:,}")
print(f"   Excessive special chars:      {spam_counts['special_char_spam']:,}")
print(f"   Suspiciously high scores:     {spam_counts['high_score_spam']:,}")

total_spam = spam_counts['char_spam'] + spam_counts['special_char_spam'] + spam_counts['high_score_spam']

if total_spam == 0:
    print("\n✅ No obvious spam patterns detected")
else:
    print(f"\n⚠️ Potential spam indicators found: {total_spam}")
    
print("="*80)

In [0]:
print("\n🗑️ DEDUPLICATION - REMOVE DUPLICATE POSTS")
print("="*80)

# Count duplicates before removal
initial_count = df_with_bot_flag.count()
print(f"\n📊 Before deduplication: {initial_count:,} records")

# Use window function to keep the latest record for each ID
window_spec = Window.partitionBy("id").orderBy(F.col("load_date").desc())

df_silver = df_with_bot_flag \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

final_count = df_silver.count()
duplicates_removed = initial_count - final_count

print(f"\n📊 After deduplication:  {final_count:,} records")
print(f"   Duplicates removed:       {duplicates_removed:,}")

if duplicates_removed == 0:
    print("\n✅ No duplicates found")
else:
    print(f"\n✅ Deduplication complete - removed {duplicates_removed:,} duplicate records")
    
print("="*80)

In [0]:
# Write to Silver layer Delta table
silver_table = "workspace.redditrecon.posts_silver"

print(f"\n📦 WRITING TO SILVER LAYER")
print("="*80)

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print(f"✅ Successfully written to: {silver_table}")
print("   Mode: overwrite")
print("   Format: Delta")
print("="*80)

In [0]:
print("\n📊 SILVER LAYER VALIDATION & METRICS")
print("="*80)

# Read back from Silver table to validate
df_silver_check = spark.table(silver_table)

# Calculate metrics
metrics = df_silver_check.agg(
    F.count("*").alias("total_records"),
    F.sum(F.when(F.col("is_bot"), 1).otherwise(0)).alias("bot_records"),
    F.countDistinct("id").alias("unique_posts"),
    F.countDistinct("author").alias("unique_authors"),
    F.countDistinct("subreddit").alias("unique_subreddits"),
    F.min("created_at").alias("earliest_post"),
    F.max("created_at").alias("latest_post")
).collect()[0]

# Display metrics
print(f"\n📊 SUMMARY METRICS:")
print(f"   Total Records:        {metrics['total_records']:,}")
print(f"   Unique Posts:         {metrics['unique_posts']:,}")
print(f"   Unique Authors:       {metrics['unique_authors']:,}")
print(f"   Unique Subreddits:    {metrics['unique_subreddits']:,}")

print(f"\n🤖 BOT DETECTION:")
print(f"   Bot Records:          {metrics['bot_records']:,} ({metrics['bot_records']/metrics['total_records']*100:.2f}%)")
print(f"   Non-Bot Records:      {metrics['total_records'] - metrics['bot_records']:,} ({(metrics['total_records']-metrics['bot_records'])/metrics['total_records']*100:.2f}%)")

print(f"\n📅 DATE RANGE:")
print(f"   Earliest Post:        {metrics['earliest_post']}")
print(f"   Latest Post:          {metrics['latest_post']}")

print("\n" + "="*80)
print("✅ SILVER LAYER VALIDATION COMPLETE!")
print("\n🚀 Silver table ready for Gold layer transformations")
print("="*80)

df_silver_check.display()